# 🇵🇰 Pakistan Economic Indicators — Exploratory Data Analysis
## Project: GDP Growth Forecasting & Economic Health Dashboard (2000–2025)

**Problem Statement**  
Pakistan's macroeconomic trajectory is shaped by a complex interplay of domestic fiscal policy, external shocks, and structural factors. This project builds a reproducible ML pipeline to **forecast annual GDP growth** and derive an **economic health score** — providing actionable insight for policymakers, investors, and researchers.

**Objectives**
1. Understand the distribution and temporal dynamics of 30+ economic indicators
2. Identify features most strongly correlated with GDP growth
3. Detect regime changes (crises, boom periods) through data patterns
4. Inform feature engineering decisions for downstream modeling

**Success Criteria**
- RMSE < 1.5% on held-out test years for GDP growth prediction
- All key patterns documented with domain-aligned interpretation
- Reproducible pipeline (seed-controlled, path-agnostic)

## Section 1 — Environment Setup & Project Structure

In [ ]:
import sys, os, warnings, logging
from pathlib import Path

# ── Make project root importable ──────────────────────────────────────────
ROOT = Path.cwd().parent          # notebooks/ → project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
from IPython.display import display

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Style ─────────────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 130, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

# ── Paths ─────────────────────────────────────────────────────────────────
import config
FIGURES_DIR = config.FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Environment ready  |  ROOT:", ROOT)
print("   Figures →", FIGURES_DIR)

## Section 2 — Data Acquisition & Storage

In [ ]:
from src.data import load_raw, basic_info

# ── Load raw dataset ──────────────────────────────────────────────────────
df = load_raw()

info = basic_info(df)
print(f"Shape      : {info['shape']}")
print(f"Year range : {info['year_range']}")
print(f"\nMissing values:")
missing = {k: v for k, v in info['missing_counts'].items() if v > 0}
if missing:
    for k, v in missing.items():
        print(f"  {k}: {v}")
else:
    print("  ✅ No missing values")

display(df.head())
display(df.describe().round(2))

## Section 3 — Data Cleaning & Preprocessing Pipeline

In [ ]:
from src.data.preprocess import cap_outliers, impute_missing
from src.features import build_all_features

# ── Check for outliers (IQR method) ──────────────────────────────────────
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
Q1, Q3 = df[numeric_cols].quantile(0.25), df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((df[numeric_cols] < (Q1 - 1.5 * IQR)) |
                  (df[numeric_cols] > (Q3 + 1.5 * IQR))).sum()
print("Outlier counts per column:")
display(outlier_counts[outlier_counts > 0].sort_values(ascending=False).to_frame("outliers"))

# ── Apply winsorization + feature engineering ─────────────────────────────
df_clean  = cap_outliers(df, numeric_cols, lower=0.02, upper=0.98)
df_feat   = build_all_features(df_clean)

print(f"\nData after feature engineering: {df_feat.shape}")

# ── Persist processed features ────────────────────────────────────────────
config.DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)
df_feat.to_csv(config.PROCESSED_FILE)
print(f"✅ Processed data saved → {config.PROCESSED_FILE}")

## Section 4 — Exploratory Data Analysis & Visualization

### 4.1 GDP Timeline

In [ ]:
from src.visualization.plots import (
    plot_gdp_timeline, plot_correlation_heatmap,
    plot_macro_panel, plot_external_sector, plot_distributions
)

# ── 4.1: GDP Timeline ─────────────────────────────────────────────────────
fig = plot_gdp_timeline(df, save_path=FIGURES_DIR / "gdp_timeline.png")
plt.show()

# Key observations
print("""
📌 Key Observations:
  • 2005: Peak growth of 9.0% — commodity boom + CPEC precursor
  • 2020: First contraction (-0.5%) since 2001 — COVID-19
  • 2023: Near-recession (+−0.04%) — twin-deficit + PKR crash
  • 2025: Recovery trajectory (3.0%) — IMF stabilisation + record remittances
""")

### 4.2 Macro Panel & External Sector

In [ ]:
fig = plot_macro_panel(df, save_path=FIGURES_DIR / "macro_panel.png")
plt.show()

fig = plot_external_sector(df, save_path=FIGURES_DIR / "external_sector.png")
plt.show()

### 4.3 Correlation Analysis

In [ ]:
core_cols = [
    "gdp_growth_pct", "inflation_cpi_pct", "unemployment_pct",
    "policy_rate_pct", "pkr_per_usd", "public_debt_gdp_pct",
    "forex_reserves_usd_bn", "remittances_usd_bn",
    "exports_usd_bn", "imports_usd_bn",
    "current_account_usd_bn", "fdi_inflows_usd_bn",
    "tax_revenue_gdp_pct", "mobile_per_100",
]
fig = plot_correlation_heatmap(
    df, cols=core_cols, save_path=FIGURES_DIR / "correlation_heatmap.png"
)
plt.show()

# Top correlates with GDP growth
corr_with_gdp = df[core_cols].corr()["gdp_growth_pct"].drop("gdp_growth_pct")
print("\n📊 Top correlates with GDP growth:")
display(corr_with_gdp.abs().sort_values(ascending=False).head(10).round(3))

### 4.4 Decade-wise Distribution Analysis

In [ ]:
# Reconstruct decade label from index
decade_map = {y: ("2000s" if y < 2010 else "2010s" if y < 2020 else "2020s")
              for y in df.index}
df_plot = df.copy()
df_plot["decade_label"] = df_plot.index.map(decade_map)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_violin = [
    ("gdp_growth_pct",   "GDP Growth (%)"),
    ("inflation_cpi_pct","Inflation (CPI %)"),
    ("forex_reserves_usd_bn", "FX Reserves (USD bn)"),
]
palette = {"2000s": "#1565C0", "2010s": "#2E7D32", "2020s": "#C62828"}

for ax, (col, label) in zip(axes, metrics_violin):
    sns.violinplot(data=df_plot, x="decade_label", y=col,
                   palette=palette, ax=ax, inner="quartile")
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("")

fig.suptitle("Decade-wise Distributions of Key Indicators", fontsize=13,
             fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "decade_distributions.png", bbox_inches="tight")
plt.show()

# Decade summary table
decade_summary = df_plot.groupby("decade_label")[
    ["gdp_growth_pct", "inflation_cpi_pct", "unemployment_pct",
     "remittances_usd_bn", "forex_reserves_usd_bn"]
].agg(["mean", "std"]).round(2)
print("\n📋 Decade Summary Statistics:")
display(decade_summary)

### 4.5 Time-Series Stationarity Tests (ADF)

In [ ]:
"""
Augmented Dickey-Fuller (ADF) test for stationarity.
H₀: unit root present (non-stationary)
p < 0.05 → reject H₀ → series is stationary
"""
test_cols = ["gdp_growth_pct", "inflation_cpi_pct", "pkr_per_usd",
             "forex_reserves_usd_bn", "remittances_usd_bn"]

adf_results = []
for col in test_cols:
    series = df[col].dropna()
    adf_stat, p_val, _, _, crit, _ = adfuller(series, autolag="AIC")
    adf_results.append({
        "Feature":   col,
        "ADF Stat":  round(adf_stat, 4),
        "p-value":   round(p_val, 4),
        "Stationary": "✅ Yes" if p_val < 0.05 else "❌ No (non-stationary)",
        "Crit 5%":   round(crit["5%"], 4),
    })

adf_df = pd.DataFrame(adf_results).set_index("Feature")
print("📈 Augmented Dickey-Fuller Stationarity Test Results:")
display(adf_df)

print("""
💡 Interpretation:
   Non-stationary series (e.g., pkr_per_usd, remittances) should be
   differenced or transformed (log / pct_change) before use in ARIMA models.
   For tree-based models, raw values are acceptable.
""")

## Section 5 — Feature Engineering & Selection

### 5.1 Engineered Features Overview

In [ ]:
print("Engineered features and their economic rationale:\n")
feature_rationale = {
    "gdp_growth_lag1":      "Autoregressive momentum — last year predicts this year",
    "gdp_growth_lag2":      "Two-year memory in growth cycles",
    "inflation_lag1":       "Lagged inflation impacts current-year consumption",
    "gdp_growth_ma3":       "3-year rolling trend smooths short-term volatility",
    "inflation_ma3":        "Trend inflation for monetary policy context",
    "pkr_yoy_change":       "YoY depreciation: import cost pressure signal",
    "forex_months_import":  "FX import cover: IMF crisis threshold = 3 months",
    "trade_openness":       "(Exports+Imports)/GDP — exposure to global shocks",
    "ext_pressure_index":   "Trade deficit − FX reserves: composite stress gauge",
    "remittances_growth":   "Growth rate of Pakistan's key foreign-exchange earner",
    "fdi_growth":           "Investment confidence signal",
}
for feat, rationale in feature_rationale.items():
    print(f"  {feat:<28} → {rationale}")

print(f"\nEngineered feature matrix shape: {df_feat.shape}")
display(df_feat[list(feature_rationale.keys())].tail(5).round(2))

### 5.2 Mutual Information Feature Selection

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import RobustScaler

# Prepare feature matrix
target_col   = config.TARGET_REG
feature_cols = [c for c in df_feat.columns
                if c not in [target_col, "gdp_growth_category",
                             "inflation_category", "decade", "key_events"]]
feature_cols = [c for c in feature_cols if df_feat[c].dtype != object]

X = df_feat[feature_cols].fillna(0)
y = df_feat[target_col]

# Mutual information
mi_scores = mutual_info_regression(X, y, random_state=SEED)
mi_df = (pd.Series(mi_scores, index=feature_cols)
           .sort_values(ascending=False)
           .head(20))

fig, ax = plt.subplots(figsize=(10, 7))
mi_df.sort_values().plot.barh(ax=ax, color="#1565C0", alpha=0.8)
ax.set_title("Mutual Information Scores vs. GDP Growth", fontsize=13,
             fontweight="bold")
ax.set_xlabel("MI Score")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "mutual_information.png", bbox_inches="tight")
plt.show()

print(f"\n🏆 Top 10 features by Mutual Information:")
display(mi_df.head(10).round(4).to_frame("MI Score"))

## Section 6 — Model Development & Hyperparameter Tuning

In [ ]:
from src.data.preprocess import temporal_split
from src.models.train import (
    build_regression_models, cross_validate_models,
    tune_random_forest, train_final_model, save_model,
)
from src.models.evaluate import (
    regression_report, compare_models,
    get_feature_importance, plot_actual_vs_predicted,
)

# ── Prepare X, y ──────────────────────────────────────────────────────────
drop_cols = ["gdp_growth_category", "inflation_category", "decade",
             "key_events", "gdp_usd_bn", "gdp_per_capita_usd",
             "imf_program_active"]
feat_matrix = df_feat.drop(columns=[c for c in drop_cols if c in df_feat.columns])
feat_matrix = feat_matrix.select_dtypes(include=np.number)
feat_matrix = feat_matrix.fillna(0)

X   = feat_matrix.drop(columns=[target_col])
y   = feat_matrix[target_col]
feature_names = X.columns.tolist()

# ── Time-aware split (last 5 years = test) ────────────────────────────────
X_train, X_test = X.iloc[:-5], X.iloc[-5:]
y_train, y_test = y.iloc[:-5], y.iloc[-5:]

print(f"Training set : {X_train.shape[0]} years ({X_train.index[0]}–{X_train.index[-1]})")
print(f"Test set     : {X_test.shape[0]}  years ({X_test.index[0]}–{X_test.index[-1]})")
print(f"Features     : {len(feature_names)}")

In [ ]:
# ── Cross-validate all baseline models ───────────────────────────────────
print("⏳ Cross-validating all models (TimeSeriesSplit, k=5)…\n")
models = build_regression_models()
cv_results = cross_validate_models(models, X_train, y_train, n_splits=5)

print("\n📊 Cross-Validation Results (RMSE — lower is better):")
display(cv_results.sort_values("mean_score").round(4))

In [ ]:
# ── Hyperparameter tuning (RandomForest) ─────────────────────────────────
print("⚙️  Grid-searching RandomForest hyper-parameters…\n")
best_rf = tune_random_forest(X_train, y_train, n_splits=5)
print(f"\n✅ Best pipeline: {best_rf}")

# ── Gradient Boosting (baseline, fast) ───────────────────────────────────
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

gb_pipe = Pipeline([
    ("scaler", RobustScaler()),
    ("model",  GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.08, max_depth=3,
        subsample=0.85, random_state=SEED
    ))
])
gb_pipe.fit(X_train, y_train)

## Section 7 — Model Validation & Evaluation

In [ ]:
all_results = {}

for name, pipe in [("RandomForest_Tuned", best_rf), ("GradientBoosting", gb_pipe)]:
    y_pred = pipe.predict(X_test)
    metrics = regression_report(y_test, y_pred, model_name=name)
    all_results[name] = metrics

# ── Add Ridge & Lasso for comparison ─────────────────────────────────────
for name, pipe in [("Ridge", models["Ridge"]), ("Lasso", models["Lasso"])]:
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    metrics = regression_report(y_test, y_pred, model_name=name)
    all_results[name] = metrics

print("\n🏆 Model Comparison (Test Set):")
comparison_df = compare_models(all_results)
display(comparison_df.round(3))

best_model_name = comparison_df["RMSE"].idxmin()
print(f"\n🥇 Best model: {best_model_name}  (RMSE = {comparison_df.loc[best_model_name, 'RMSE']:.3f})")

In [ ]:
# ── Actual vs Predicted plot ───────────────────────────────────────────────
best_pipe = best_rf if best_model_name == "RandomForest_Tuned" else gb_pipe
y_pred_best = best_pipe.predict(X_test)

fig = plot_actual_vs_predicted(
    y_test.values, y_pred_best, years=X_test.index.tolist(),
    title=f"Actual vs Predicted GDP Growth — {best_model_name}",
    save_path=FIGURES_DIR / "actual_vs_predicted.png"
)
plt.show()

# ── Feature importance ───────────────────────────────────────────────────
fi_df = get_feature_importance(best_pipe, feature_names, top_n=15)
from src.models.evaluate import plot_feature_importance
fig = plot_feature_importance(
    fi_df, title=f"Top 15 Features — {best_model_name}",
    save_path=FIGURES_DIR / "feature_importance.png"
)
plt.show()

## Section 8 — Model Serialization & Experiment Tracking

In [ ]:
import json, hashlib

# ── Save best model ───────────────────────────────────────────────────────
model_path = config.MODELS_DIR / "best_model.pkl"
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
save_model(best_pipe, model_path)

# ── Save feature list ─────────────────────────────────────────────────────
feat_path = config.MODELS_DIR / "feature_names.json"
with open(feat_path, "w") as f:
    json.dump(feature_names, f, indent=2)

# ── Create experiment manifest ───────────────────────────────────────────
with open(model_path, "rb") as f:
    model_hash = hashlib.md5(f.read()).hexdigest()

manifest = {
    "model_name":   best_model_name,
    "trained_on":   f"{X_train.index[0]}–{X_train.index[-1]}",
    "test_years":   X_test.index.tolist(),
    "features":     len(feature_names),
    "metrics":      all_results[best_model_name],
    "model_md5":    model_hash,
    "python":       sys.version.split()[0],
}
import json
manifest_path = config.MODELS_DIR / "experiment_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("✅ Saved:")
print(f"   Model     → {model_path}")
print(f"   Features  → {feat_path}")
print(f"   Manifest  → {manifest_path}")
print(f"\n📋 Experiment Manifest:")
print(json.dumps(manifest, indent=2))

## Section 9 — Deployment: Scenario-Based Forecasting

Using the trained model to generate forward-looking policy scenarios.
This pattern mirrors what a central bank or IMF analyst would do.

In [ ]:
from src.models.predict import scenario_forecast

# Base case: approximate 2025 values (last row of training)
base_row = X.iloc[-1].to_dict()

scenarios = {
    "Baseline (2026)":      {},                                            # No change
    "Optimistic":           {"inflation_cpi_pct": 5.0,
                             "remittances_growth": 15.0,
                             "forex_months_import": 3.5,
                             "pkr_yoy_change": 2.0},
    "Pessimistic":          {"inflation_cpi_pct": 20.0,
                             "remittances_growth": -5.0,
                             "forex_months_import": 1.5,
                             "pkr_yoy_change": 25.0},
    "IMF Program Active":   {"imf_program_active": 1,
                             "policy_rate_pct": 13.0,
                             "ext_pressure_index": -0.1},
    "Record Remittances":   {"remittances_growth": 30.0,
                             "remittances_gdp_pct": 11.0},
}

scenario_df = scenario_forecast(
    base_row      = base_row,
    scenarios     = scenarios,
    model_path    = config.MODELS_DIR / "best_model.pkl",
)
print("📊 2026 GDP Growth Forecast by Scenario:\n")
display(scenario_df[["predicted_gdp_growth"]].round(2))

# Visual
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#1565C0" if v >= 3 else "#E53935" if v < 1.5 else "#F57F17"
          for v in scenario_df["predicted_gdp_growth"]]
scenario_df["predicted_gdp_growth"].plot.barh(ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("2026 GDP Growth Forecast — Scenario Analysis", fontsize=13,
             fontweight="bold")
ax.set_xlabel("Predicted GDP Growth (%)")
for i, v in enumerate(scenario_df["predicted_gdp_growth"]):
    ax.text(v + 0.1, i, f"{v:.2f}%", va="center", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "scenario_forecast.png", bbox_inches="tight")
plt.show()